# A Dynamic Model of Systemic Collapse During Heat Waves

## Last updated: 2025-03-24

## This notebook simulates how systemic collapse occurs during extreme heat events, using the British Columbia Heat Dome as case study.

## This notebook is divided into 3 parts:

### 1. Simulation graph
### 2. Asymptote graph
### 3. Heat wave graph

## 1. Simulation graph

### We use scipy.integrate.odeint to run the dynamic model over a 30-day period, comparing a linear scenario versus collapse scenario.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

plt.style.use('classic')

def heat_dynamics(y, t, beta, alpha, rho, gamma, delta):
    S, I, H = y

    flow_S_to_I = beta * S * (1 + alpha * I)
    flow_I_to_S = rho * I
    flow_I_to_H = gamma * I
    flow_H_to_S = delta * H

    return [-flow_S_to_I + flow_I_to_S + flow_H_to_S,
             flow_S_to_I - flow_I_to_S - flow_I_to_H,
             flow_I_to_H - flow_H_to_S]

t = np.linspace(0, 30, 300)

sol_linear = odeint(heat_dynamics, [99, 1, 0], t, args=(0.05, 0.0, 0.1, 0.05, 0.1))
sol_chaos  = odeint(heat_dynamics, [99, 1, 0], t, args=(0.05, 0.05, 0.1, 0.05, 0.1))

plt.figure(figsize=(7, 5))
plt.plot(t, sol_linear[:, 1], 'b-', linewidth=2)
plt.plot(t, sol_chaos[:, 1], 'r-', linewidth=2,)
plt.xlabel('Days', fontsize=12)
plt.ylabel('$I$', fontsize=12)
plt.savefig('simplot_updated.png', bbox_inches='tight', dpi=300)
plt.close()

## 2. Asymptote graph

### We calculate the tipping point of the system, plotting exposure against systemic burden to reveal the asymptote.

In [3]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('classic')

N = 100
rho = 0.1
gamma = 0.05
alpha = 0.01

beta_collapse = (rho + gamma) / (N * alpha)

beta_values_red = np.linspace(0, beta_collapse - 0.001, 500)
I_steady_red = (beta_values_red * N) / ((rho + gamma) - (beta_values_red * N * alpha))

beta_values_blue = np.linspace(0, 0.20, 500)
I_steady_blue = (beta_values_blue * N) / (rho + gamma)

plt.plot(beta_values_blue, I_steady_blue, 'b-', linewidth=2)
plt.plot(beta_values_red, I_steady_red, 'r-', linewidth=3)
plt.axvline(x=beta_collapse, color='black', linestyle='--', linewidth=2)

plt.xlabel('β', fontsize=12)
plt.ylabel('$I_{steady}$', fontsize=12)
plt.ylim(0, 300)
plt.xlim(0, 0.20)
plt.savefig('asymptote.png', bbox_inches='tight', dpi=300)
plt.close()

## 3. BC graph

### We use scipy.optimize.curve_fit to map theoretical models against the 2021 BC Heat Dome mortality data, pinpointing the exact temperature of collapse.

### Data extracted from the Climate Change Institute report (p. 10).

### Link: https://climateinstitute.ca/wp-content/uploads/2023/06/The-case-for-adapting-to-extreme-heat-costs-of-the-BC-heat-wave.pdf

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 12,
    "axes.linewidth": 1.2,
    "lines.linewidth": 2,
    "xtick.direction": "in",
    "ytick.direction": "in"
})

temps = np.array([39.2, 43.8, 45.2, 46.6, 47.9])
deaths = np.array([9, 15, 56, 137, 234])

def model_linear(t, m, c):
    return m * t + c

def model_dynamic(t, k, t_crit):
    return np.where(t < t_crit, k / (t_crit - t), np.inf)

popt_lin, _ = curve_fit(model_linear, temps, deaths)

try:
    popt_asym, _ = curve_fit(model_dynamic, temps, deaths,
                             p0=[500, 48.5], bounds=([1, 48.0], [5000, 55]))
    k_fit, tc_fit = popt_asym
except RuntimeError:
    k_fit, tc_fit = 100, 50

t_smooth = np.linspace(38, tc_fit - 0.05, 1000)
y_lin = model_linear(t_smooth, *popt_lin)
y_asym = model_dynamic(t_smooth, k_fit, tc_fit)

fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(temps, deaths, color='k', s=80, label='Observations (June25-29)', zorder=10)

ax.plot(t_smooth, y_asym, 'r-', label=r'Dynamic model ($\alpha>0$)')

ax.plot(t_smooth, y_lin, 'b-', label=r'Linear model ($\alpha=0$)')

ax.axvline(x=tc_fit, color='black', linestyle='--', linewidth=2)

ax.set_xlabel(r'Maximum Temperature ($^\circ\mathrm{C}$)')
ax.set_ylabel(r'Daily Deaths')
ax.set_ylim(-20, 350)
ax.set_xlim(38, 51)

ax.legend(frameon=False, loc='upper left')

plt.savefig("heat_dome_dynamic.png", dpi=300, transparent=True, bbox_inches='tight')
plt.close()